# Azure Data Engineering Project

## Objective
Build an end-to-end Azure Data Engineering pipeline using:

- Azure Data Factory
- Azure Data Lake Storage Gen2
- Azure Databricks
- Unity Catalog
- Delta Lake
- PySpark

## Architecture

Landing → Bronze → Silver → Gold


## 1. Imports

In [ ]:
from pyspark.sql.functions import col, trim, initcap, count, avg

## 2. Configuration

In [ ]:
storage_account = "project1storageaccount"

landing_path = f"abfss://landing@{storage_account}.dfs.core.windows.net/customers.csv"
bronze_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/customers"
silver_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/customers"
gold_path = f"abfss://gold@{storage_account}.dfs.core.windows.net/customer_summary"


## 3. Landing → Bronze

Read the raw CSV from the Landing container and store it as a Delta table in the Bronze layer.

In [0]:
spark

In [0]:
df=spark.read \
    .option("header","true") \
    .option("inferschema","true") \
    .csv("abfss://landing@project1storageaccount.dfs.core.windows.net/customers.csv")
df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("abfss://bronze@project1storageaccount.dfs.core.windows.net/customers")

In [0]:
from pyspark.sql.functions import trim ,initcap, col 
bronze_df= spark.read.format('delta').load(

    "abfss://bronze@project1storageaccount.dfs.core.windows.net/customers"
)
display(bronze_df)
silver_df = bronze_df.dropDuplicates()
silver_df = silver_df.withColumn(
    "City",
    trim(col("City"))
)
silver_df = silver_df.withColumn(
    "City",
    initcap(col("City"))
)
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("abfss://silver@project1storageaccount.dfs.core.windows.net/customers")


customer_id,customer_name,city,state,email
101,Alice,Hyderabad,TS,alice@test.com
102,Bob,Bangalore,KA,bob@test.com
103,Charlie,Chennai,TN,charlie@test.com
104,David,Mumbai,MH,david@test.com
105,Eva,Pune,MH,eva@test.com
106,Frank,Delhi,DL,frank@test.com
107,Grace,Hyderabad,TS,grace@test.com
108,Helen,Chennai,TN,helen@test.com


In [0]:
gold_df = spark.read.format("delta").load("abfss://silver@project1storageaccount.dfs.core.windows.net/customers")
display(gold_df)
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("abfss://gold@project1storageaccount.dfs.core.windows.net/customer_summary")

customer_id,customer_name,City,state,email
104,David,Mumbai,MH,david@test.com
103,Charlie,Chennai,TN,charlie@test.com
102,Bob,Bangalore,KA,bob@test.com
106,Frank,Delhi,DL,frank@test.com
107,Grace,Hyderabad,TS,grace@test.com
101,Alice,Hyderabad,TS,alice@test.com
105,Eva,Pune,MH,eva@test.com
108,Helen,Chennai,TN,helen@test.com


## 4. Validation

In [ ]:
print("Pipeline executed successfully!")

# Optional validation
# bronze_df.count()
# silver_df.count()
# gold_df.count()


## 5. Conclusion

### Technologies Used

- Azure Data Factory
- Azure Data Lake Storage Gen2
- Azure Databricks
- Unity Catalog
- Delta Lake
- PySpark

### Outcome

Successfully implemented a Medallion Architecture:
Landing → Bronze → Silver → Gold
